# Bronze → Silver — Microdados INEP

Este notebook executa o pipeline de limpeza e normalização dos Microdados do Censo Escolar.

**O que ele faz:**
- Lê todos os anos de `data/bronze/ano=YYYY/microdados_ed_basica_YYYY.csv`
- Seleciona ~60 colunas relevantes e renomeia para `snake_case` em português
- Converte tipos (int, str), preenche nulos e padroniza indicadores (0/1)
- Salva em `data/silver/microdados_escola/ano=YYYY/microdados_escola.parquet`


## Passo 1: Importação de Módulos

In [ ]:
import sys
import os

# Garante que o diretório raiz do projeto está no path
project_root = os.path.abspath(os.path.join(os.getcwd(), '..', '..'))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

from src.jobs.silver.bronze_to_silver import (
    run, COLUMN_RENAME_MAP, INDICATOR_COLS, QUANTITY_COLS
)
from src.config import settings
from src.utils.logger import get_logger
import pandas as pd
from pathlib import Path

logger = get_logger('notebook_silver')
print('✓ Módulos carregados com sucesso')

## Passo 2: Inspecionar Colunas Disponíveis no Bronze (2024)

In [ ]:
# Leitura de amostra do Bronze mais recente para inspecionar
bronze_path = Path(settings.BRONZE_DATA_PATH)
sample_file = bronze_path / 'ano=2024' / 'microdados_ed_basica_2024.csv'

df_sample = pd.read_csv(sample_file, sep=';', encoding='latin-1', nrows=5, dtype=str)
print(f'Colunas no arquivo 2024: {len(df_sample.columns)}')
print(f'\nColunas mapeadas para Silver: {len(COLUMN_RENAME_MAP)}')
print('\nMapeamento (Bronze → Silver):')
for orig, novo in COLUMN_RENAME_MAP.items():
    disponivel = '✓' if orig in df_sample.columns else '✗'
    print(f'  {disponivel} {orig:45s} → {novo}')

## Passo 3: Executar Pipeline Bronze → Silver

> ⚠️ **Este processo pode demorar alguns minutos** pois processa todos os anos (2007–2024).

In [ ]:
run()

## Passo 4: Verificação da Camada Silver

In [ ]:
silver_base = Path(settings.SILVER_DATA_PATH) / 'microdados_escola'
parquet_files = sorted(silver_base.glob('ano=*/microdados_escola.parquet'))

print(f'Arquivos Silver gerados: {len(parquet_files)}\n')
print(f'{"Ano":<8} {"Registros":>12} {"Colunas":>10} {"Tamanho (KB)":>14}')
print('-' * 50)

total = 0
for f in parquet_files:
    df_v = pd.read_parquet(f, engine='pyarrow')
    size_kb = f.stat().st_size // 1024
    total += len(df_v)
    ano = f.parent.name
    print(f'{ano:<8} {len(df_v):>12,} {len(df_v.columns):>10} {size_kb:>13,}')

print('-' * 50)
print(f'{"TOTAL":<8} {total:>12,}')

In [ ]:
# Inspeciona o schema do arquivo mais recente
df_last = pd.read_parquet(silver_base / 'ano=2024' / 'microdados_escola.parquet')
print('Schema da Silver (ano=2024):')
print(df_last.dtypes.to_string())
print(f'\nAmostra (5 primeiras linhas):')
df_last.head()